# Brief Inspection — CIC-UNSW-NB15

**Notebook 04 of EDA series**
**Author:** Bradly Morein (UP2088701)
**Date:** 29 April 2026

## Purpose

Confirm that CIC-UNSW-NB15 (the secondary cross-dataset validation set per 
ADR 0002) has the structure we expect:

1. **Schema match with CIC-IDS2017** — same CICFlowMeter feature columns, 
   so models trained on the primary dataset can be applied to this secondary 
   dataset without feature alignment.
2. **Label encoding** — `Label.csv` contains integers 0–9 per the Readme, 
   not strings.
3. **Row counts and class distribution** — 80/20 benign:attack as per the 
   CIC team's documentation.

This is a fast confirmation pass, not full EDA. Full secondary-dataset 
characterisation will be performed in Week 7 when the cross-dataset 
evaluation is run.

## Data source

- **Dataset:** CIC-UNSW-NB15 (Mohammadian, Lashkari & Ghorbani 2024)
- **Acquisition:** registered browser session, 29 April 2026
- **SHA-256 record:** `data/sha256sums.txt`
- **ADR for dataset choice:** `docs/decisions/0002-secondary-dataset.md`

In [1]:
"""Imports and notebook-wide settings."""
from pathlib import Path

import numpy as np
import pandas as pd

REPO_ROOT = Path.cwd()
while not (REPO_ROOT / ".git").exists() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent

UNSW_DIR    = REPO_ROOT / "data" / "raw" / "cic-unsw-nb15"
CICIDS_DIR  = REPO_ROOT / "data" / "interim" / "cicids2017-engelen"

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 200)

print(f"Repo root:       {REPO_ROOT}")
print(f"UNSW directory:  {UNSW_DIR}")
print(f"CICIDS directory: {CICIDS_DIR}")
print(f"\nUNSW files: {sorted(p.name for p in UNSW_DIR.iterdir())}")

Repo root:       /home/brad/projects/dissertation
UNSW directory:  /home/brad/projects/dissertation/data/raw/cic-unsw-nb15
CICIDS directory: /home/brad/projects/dissertation/data/interim/cicids2017-engelen

UNSW files: ['CICFlowMeter_out.csv', 'Data.csv', 'Label.csv', 'Readme.txt']


## Step 1 — Load Data.csv and check basic structure

`Data.csv` is the 80/20 balanced subset (~448k flows). We're checking:
- Shape (rows × columns).
- Memory footprint.
- Whether labels are stored in this file or only in the separate `Label.csv`.
- First few rows to confirm parseable, sensible-looking data.

In [2]:
"""Load CIC-UNSW-NB15 Data.csv and inspect structure."""
data_path = UNSW_DIR / "Data.csv"

# Pandas will infer dtypes on first read. low_memory=False suppresses 
# mixed-dtype warnings on initial inspection.
df_unsw = pd.read_csv(data_path, low_memory=False)

print(f"Shape: {df_unsw.shape[0]:,} rows × {df_unsw.shape[1]} columns")
print(f"Memory: {df_unsw.memory_usage(deep=True).sum() / 1024**2:.0f} MB\n")

print(f"Last 5 column names (to check whether label is in this file):")
for col in df_unsw.columns[-5:]:
    print(f"  {col}")

print(f"\nFirst 2 rows of first 8 columns:")
print(df_unsw.iloc[:2, :8].to_string())

Shape: 447,915 rows × 76 columns
Memory: 260 MB

Last 5 column names (to check whether label is in this file):
  Active Min
  Idle Mean
  Idle Std
  Idle Max
  Idle Min

First 2 rows of first 8 columns:
   Flow Duration  Total Fwd Packet  Total Bwd packets  Total Length of Fwd Packet  Total Length of Bwd Packet  Fwd Packet Length Max  Fwd Packet Length Min  Fwd Packet Length Mean
0         214392                 9                 21                       388.0                     24564.0                  194.0                    0.0               43.111111
1        2376792                 9                  3                       752.0                         0.0                  188.0                    0.0               83.555556


## Step 2 — Schema comparison with CIC-IDS2017

Confirm exactly which columns are shared and which differ. This determines 
which feature subset to use for cross-dataset evaluation in Week 7.

In [3]:
"""Compare column sets between CIC-UNSW-NB15 and CIC-IDS2017."""

# Load Monday from CIC-IDS2017 (any day works for column comparison; they're identical)
df_cicids = pd.read_parquet(CICIDS_DIR / "Monday-WorkingHours.parquet")

unsw_cols = set(df_unsw.columns)
cicids_cols = set(df_cicids.columns)

shared       = unsw_cols & cicids_cols
unsw_only    = unsw_cols - cicids_cols
cicids_only  = cicids_cols - unsw_cols

print(f"CIC-UNSW-NB15 columns:    {len(unsw_cols)}")
print(f"CIC-IDS2017 columns:      {len(cicids_cols)}")
print(f"Shared columns:           {len(shared)}")
print(f"Only in CIC-UNSW-NB15:    {len(unsw_only)}")
print(f"Only in CIC-IDS2017:      {len(cicids_only)}")

print(f"\n=== Shared columns ({len(shared)}) — these align across datasets ===")
for col in sorted(shared):
    print(f"  {col}")

print(f"\n=== Only in CIC-IDS2017 ({len(cicids_only)}) — must drop for cross-dataset eval ===")
for col in sorted(cicids_only):
    print(f"  {col}")

print(f"\n=== Only in CIC-UNSW-NB15 ({len(unsw_only)}) — won't have CIC-IDS2017 training data ===")
for col in sorted(unsw_only):
    print(f"  {col}")

CIC-UNSW-NB15 columns:    76
CIC-IDS2017 columns:      84
Shared columns:           76
Only in CIC-UNSW-NB15:    0
Only in CIC-IDS2017:      8

=== Shared columns (76) — these align across datasets ===
  ACK Flag Count
  Active Max
  Active Mean
  Active Min
  Active Std
  Average Packet Size
  Bwd Bulk Rate Avg
  Bwd Bytes/Bulk Avg
  Bwd Header Length
  Bwd IAT Max
  Bwd IAT Mean
  Bwd IAT Min
  Bwd IAT Std
  Bwd IAT Total
  Bwd Init Win Bytes
  Bwd PSH Flags
  Bwd Packet Length Max
  Bwd Packet Length Mean
  Bwd Packet Length Min
  Bwd Packet Length Std
  Bwd Packet/Bulk Avg
  Bwd Packets/s
  Bwd Segment Size Avg
  Bwd URG Flags
  CWR Flag Count
  Down/Up Ratio
  ECE Flag Count
  FIN Flag Count
  FWD Init Win Bytes
  Flow Bytes/s
  Flow Duration
  Flow IAT Max
  Flow IAT Mean
  Flow IAT Min
  Flow IAT Std
  Flow Packets/s
  Fwd Act Data Pkts
  Fwd Bulk Rate Avg
  Fwd Bytes/Bulk Avg
  Fwd Header Length
  Fwd IAT Max
  Fwd IAT Mean
  Fwd IAT Min
  Fwd IAT Std
  Fwd IAT Total
  Fwd PSH 

## Step 3 — Verify Label.csv structure

Per the Readme, `Label.csv` should contain integers 0–9 mapped as follows:

| Code | Class |
|---:|---|
| 0 | Benign |
| 1 | Analysis |
| 2 | Backdoor |
| 3 | DoS |
| 4 | Exploits |
| 5 | Fuzzers |
| 6 | Generic |
| 7 | Reconnaissance |
| 8 | Shellcode |
| 9 | Worms |

We confirm row count alignment with `Data.csv` and the class distribution.

In [4]:
"""Inspect Label.csv and check alignment with Data.csv."""
label_path = UNSW_DIR / "Label.csv"
df_label = pd.read_csv(label_path, header=None, names=["label"])

print(f"Label.csv shape: {df_label.shape}")
print(f"Data.csv shape:  ({len(df_unsw):,}, ...)")
print(f"Row counts match: {len(df_label) == len(df_unsw)}")

print(f"\nLabel value counts (with mapping per Readme):")
label_map = {
    0: "Benign", 1: "Analysis", 2: "Backdoor", 3: "DoS", 4: "Exploits",
    5: "Fuzzers", 6: "Generic", 7: "Reconnaissance", 8: "Shellcode", 9: "Worms",
}
counts = df_label["label"].value_counts().sort_index()
for code, count in counts.items():
    name = label_map.get(code, f"UNKNOWN-{code}")
    pct = 100 * count / len(df_label)
    print(f"  {code}  {name:<20}  {count:>8,}  ({pct:>5.2f}%)")

n_benign = (df_label["label"] == 0).sum()
n_attack = (df_label["label"] != 0).sum()
print(f"\nBenign:  {n_benign:>8,} ({100*n_benign/len(df_label):.2f}%)")
print(f"Attack:  {n_attack:>8,} ({100*n_attack/len(df_label):.2f}%)")
print(f"Imbalance ratio (benign:attack): {n_benign/n_attack:.2f}:1")

Label.csv shape: (447916, 1)
Data.csv shape:  (447,915, ...)
Row counts match: False

Label value counts (with mapping per Readme):
  0  UNKNOWN-0              358,332  (80.00%)
  1  UNKNOWN-1                  385  ( 0.09%)
  2  UNKNOWN-2                  452  ( 0.10%)
  3  UNKNOWN-3                4,467  ( 1.00%)
  4  UNKNOWN-4               30,951  ( 6.91%)
  5  UNKNOWN-5               29,613  ( 6.61%)
  6  UNKNOWN-6                4,632  ( 1.03%)
  7  UNKNOWN-7               16,735  ( 3.74%)
  8  UNKNOWN-8                2,102  ( 0.47%)
  9  UNKNOWN-9                  246  ( 0.05%)
  Label  UNKNOWN-Label                1  ( 0.00%)

Benign:         0 (0.00%)
Attack:   447,916 (100.00%)
Imbalance ratio (benign:attack): 0.00:1


In [5]:
"""Reload Label.csv correctly — header is present, not absent."""

# Issue diagnosed: Label.csv has a header row, our previous load
# interpreted the literal 'Label' header as if it were data, producing
# off-by-one row count and string-typed labels.
df_label = pd.read_csv(label_path)  # default header='infer' = first row is header

print(f"Label.csv shape: {df_label.shape}")
print(f"Data.csv shape:  ({len(df_unsw):,}, ...)")
print(f"Row counts match: {len(df_label) == len(df_unsw)}")
print(f"Label dtype: {df_label['Label'].dtype}")

print(f"\nLabel value counts (with mapping per Readme):")
label_map = {
    0: "Benign", 1: "Analysis", 2: "Backdoor", 3: "DoS", 4: "Exploits",
    5: "Fuzzers", 6: "Generic", 7: "Reconnaissance", 8: "Shellcode", 9: "Worms",
}
counts = df_label["Label"].value_counts().sort_index()
for code, count in counts.items():
    name = label_map.get(code, f"UNKNOWN-{code}")
    pct = 100 * count / len(df_label)
    print(f"  {code}  {name:<20}  {count:>8,}  ({pct:>5.2f}%)")

n_benign = (df_label["Label"] == 0).sum()
n_attack = (df_label["Label"] != 0).sum()
print(f"\nBenign:  {n_benign:>8,} ({100*n_benign/len(df_label):.2f}%)")
print(f"Attack:  {n_attack:>8,} ({100*n_attack/len(df_label):.2f}%)")
print(f"Imbalance ratio (benign:attack): {n_benign/n_attack:.2f}:1")

Label.csv shape: (447915, 1)
Data.csv shape:  (447,915, ...)
Row counts match: True
Label dtype: int64

Label value counts (with mapping per Readme):
  0  Benign                 358,332  (80.00%)
  1  Analysis                   385  ( 0.09%)
  2  Backdoor                   452  ( 0.10%)
  3  DoS                      4,467  ( 1.00%)
  4  Exploits                30,951  ( 6.91%)
  5  Fuzzers                 29,613  ( 6.61%)
  6  Generic                  4,632  ( 1.03%)
  7  Reconnaissance          16,735  ( 3.74%)
  8  Shellcode                2,102  ( 0.47%)
  9  Worms                      246  ( 0.05%)

Benign:   358,332 (80.00%)
Attack:    89,583 (20.00%)
Imbalance ratio (benign:attack): 4.00:1


## Notebook 04 summary — CIC-UNSW-NB15 fully validated

### Headline confirmations

- **Schema**: `Data.csv` has 76 columns; all 76 are present in CIC-IDS2017's
  84-column CICFlowMeter schema. The CIC team has stripped exactly the
  identifier columns we'd want to drop for cross-dataset evaluation
  (Flow ID, Src IP, Dst IP, Src Port, Dst Port, Protocol, Timestamp, Label
  — note that Label is in the separate `Label.csv` here).
- **Labels**: `Label.csv` has 447,915 integer-encoded labels (0–9) per the
  Readme mapping. Decoded class distribution exactly matches the published
  CIC-UNSW-NB15 documentation.
- **Row counts**: 447,915 in both `Data.csv` and `Label.csv`. They join
  cleanly by row index.
- **Class balance**: 80.00% benign / 20.00% attack — as documented.

### Cross-dataset experimental design implications

1. **Use the 76-feature aligned subset for primary experiments.** Train
   CIC-IDS2017 models on the same 76 columns that exist in CIC-UNSW-NB15.
   This means dropping `Src Port`, `Dst Port`, `Protocol` from CIC-IDS2017
   training (in addition to the identifier strings already dropped). 
   Slight loss of feature richness on CIC-IDS2017 traded for direct 
   cross-dataset applicability.

2. **Per-class imbalance is much milder in CIC-UNSW-NB15** (126:1 from 
   most-to-rarest class) than CIC-IDS2017 (207,000:1). The smoother 
   distribution makes CIC-UNSW-NB15 a stronger generalisation test: 
   models that essentially learned to detect specific high-volume CIC-IDS2017 
   attacks (DoS Hulk especially) should degrade more severely on the more 
   uniformly-distributed CIC-UNSW-NB15 attacks. This is exactly the kind 
   of behaviour a cross-dataset evaluation should surface.

3. **No micro-class problem in CIC-UNSW-NB15.** Smallest class is Worms 
   at 246 flows — three orders of magnitude larger than CIC-IDS2017's 
   smallest classes (Heartbleed: 11, SSH-Patator-Attempted: 8). Per-class 
   results in Week 7 will have meaningful confidence intervals for every 
   class.

### Minor methodological note

`Label.csv` has a header row (`Label`). Initial inspection cell loaded 
with `header=None`, producing an off-by-one row mismatch and string-typed 
labels. Corrected by removing the `header=None` argument. Recorded here 
as a reminder: real-world datasets have inconsistent conventions; always 
verify dtype and row count after loading rather than assuming.

### Status

CIC-UNSW-NB15 is now fully provenanced (SHA-256 hashes recorded), 
documented (ADR 0002), and structurally validated. Ready to be used as 
the secondary dataset in Week 7 cross-dataset experiments.